# 2. Storage Security — Access Control, SAS, Encryption, Protection

Azure Storage is the plumbing behind almost every Azure service — VM disks,
backups, logs, static sites, data lakes, function code. A misconfigured
storage account is one of the most common breach vectors in the cloud.

### What you'll learn
1. **Five ways to access storage** and which to prefer.
2. **SAS tokens** — build one, see the signature, compare types.
3. **Stored access policies** — revoke SAS tokens without rotating keys.
4. **Network protection** — firewall, private endpoints, "allow trusted services".
5. **Encryption** — SSE, BYOK (CMK), infrastructure encryption, double encryption.
6. **Data protection** — soft delete, versioning, immutable storage (WORM).
7. **Bad → best progression** — harden an account step by step.

### Analogy 🗄️
Think of a storage account as a shared warehouse:
- **Access keys** = the master key to the whole warehouse (avoid giving it out).
- **SAS tokens** = a time-limited visitor badge to one specific aisle.
- **Entra RBAC** = your corporate employee badge — centrally managed, revocable.
- **Private endpoint** = the warehouse door is *only* inside your office park.
- **Immutable storage** = a safe where you can put things in but nobody —
  including you — can take them out until the timer expires.

## Before you run this notebook

1. From the lab folder run `uv sync`.
2. In VS Code, pick the `.venv` kernel (top-right kernel picker).
3. Reload the window if the kernel doesn't appear.

Everything is simulated in plain Python — no Azure account required.

## 1. Five ways to authenticate to Storage

| Method | Who uses it | Exam priority | Notes |
|--------|------------|---------------|-------|
| **Entra ID + RBAC**           | Apps with managed identity, humans with Entra | ✅ Preferred   | Data-plane roles like *Storage Blob Data Reader* |
| **Shared Access Signatures**  | External partners, short-lived links          | Common         | Time + IP + permissions scoped |
| **Stored access policies + SAS** | Same as SAS, but server-side revocable     | Important      | Change policy → all SAS invalid instantly |
| **Account (access) keys**     | Legacy tools, root admin                      | ⚠️ Avoid       | Like the WiFi password of God |
| **Anonymous public access**   | Static websites, public downloads             | Off by default | Must enable at account AND container level |

> 💡 **Best practice**: disable shared-key auth and anonymous access entirely,
> then give every caller either an Entra identity or a short-lived SAS.

```bash
# Turn off the "root password" on a storage account
az storage account update -g rg-prod -n mysa \
  --allow-shared-key-access false \
  --allow-blob-public-access false \
  --default-action Deny          # network firewall default = deny
```

In [ ]:
# Simulate the decision tree: what should a given caller use?
def recommend_auth(caller):
    if caller['type'] == 'azure_app' and caller.get('has_managed_identity'):
        return '✅ Entra ID + RBAC via managed identity. No secrets anywhere.'
    if caller['type'] == 'external_partner':
        if caller.get('needs_duration_hours', 0) <= 24:
            return '✅ User-delegation SAS (signed by Entra token, not the account key).'
        return '⚠️ SAS with stored access policy so you can revoke if needed.'
    if caller['type'] == 'static_website':
        return '✅ Anonymous read on $web container (public by design).'
    if caller['type'] == 'legacy_tool' and caller.get('can_use_entra') is False:
        return '⚠️ SAS with short expiry. Plan migration — don\'t hand out account keys.'
    return '❓ Re-evaluate: what identity does the caller have?'

callers = [
    {'name': 'payroll-api',     'type': 'azure_app',         'has_managed_identity': True},
    {'name': 'partner-upload',  'type': 'external_partner',  'needs_duration_hours': 4},
    {'name': 'audit-archive',   'type': 'external_partner',  'needs_duration_hours': 720},
    {'name': 'marketing-site',  'type': 'static_website'},
    {'name': 'old-etl-tool',    'type': 'legacy_tool',       'can_use_entra': False},
]
for c in callers:
    print(f'{c["name"]:20s} → {recommend_auth(c)}')

## 2. SAS tokens — build one from scratch

A SAS token is just a **URL with a bunch of query parameters and a signature**.
The signature is `HMAC-SHA256(stringToSign, accountKey)`. Knowing how the pieces
fit together demystifies the whole system.

Three flavours:

| Type                | Signed by          | Scope                       |
|---------------------|--------------------|-----------------------------|
| **Account SAS**     | Account key        | Multiple services, broad    |
| **Service SAS**     | Account key        | One service (blob/file/…)   |
| **User-delegation SAS** | Entra user's key  | Blob service; **best choice** — ties to Entra identity, revoked if user leaves |

### How long should a SAS live?
- Ad-hoc human download: **< 1 hour**
- Partner batch upload: **24 hours**
- "Never" → don't use SAS. Use Entra + RBAC.

> ⚠️ If you rotate the account key, **all** SAS tokens signed by it stop working
> immediately. Use stored access policies for graceful revocation.

In [ ]:
# Build a SAS string-to-sign and signature, the same way the Azure SDK does.
# We use a FAKE key so nothing here is sensitive.
import base64, hmac, hashlib
from datetime import datetime, timedelta, timezone
from urllib.parse import urlencode

FAKE_ACCOUNT_KEY_B64 = base64.b64encode(b'this-is-not-a-real-key-0123456789').decode()
ACCOUNT = 'mysa'
CONTAINER = 'reports'
BLOB = 'q1-2025.csv'

def build_service_sas(permissions='r', minutes=15, ip=None):
    start  = datetime.now(timezone.utc) - timedelta(minutes=1)
    expiry = start + timedelta(minutes=minutes)
    params = {
        'sv':  '2023-11-03',
        'sr':  'b',                         # resource = blob
        'sp':  permissions,                 # r=read, w=write, d=delete, c=create, l=list
        'st':  start.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'se':  expiry.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'spr': 'https',                     # enforce HTTPS only
    }
    if ip:
        params['sip'] = ip
    canonical_resource = f'/blob/{ACCOUNT}/{CONTAINER}/{BLOB}'
    # Simplified string-to-sign (the real format has more blank lines for unused fields)
    string_to_sign = '\n'.join([
        params['sp'], params['st'], params['se'],
        canonical_resource, '', params.get('sip',''),
        params['spr'], params['sv'], params['sr'],
        '', '', '', '', '', '',            # cache-control, content-type, etc.
    ])
    key = base64.b64decode(FAKE_ACCOUNT_KEY_B64)
    sig = base64.b64encode(hmac.new(key, string_to_sign.encode('utf-8'),
                                    hashlib.sha256).digest()).decode()
    params['sig'] = sig
    url = f'https://{ACCOUNT}.blob.core.windows.net/{CONTAINER}/{BLOB}?{urlencode(params)}'
    return string_to_sign, url

print('--- Read-only, 15 minutes, HTTPS only ---')
sts, url = build_service_sas('r', minutes=15)
print('string-to-sign:\n' + sts)
print('\nSAS URL:\n' + url)

print('\n--- Read+write, locked to one partner IP ---')
_, url2 = build_service_sas('rw', minutes=60, ip='203.0.113.10')
print(url2)

print('\n💡 Notice: `sig` is deterministic — anyone with the key + parameters can reproduce it.')
print('   That\'s exactly why we must NOT hand out the account key.')

## 3. Stored access policies — revocable SAS

If you bake a **policy identifier (`si=mypolicy`)** into a SAS token, the
server looks up *mypolicy* at validation time. Change or delete the policy and
every SAS referencing it is revoked instantly — no key rotation needed.

```bash
# Create a stored access policy on a container
az storage container policy create \
  --account-name mysa -c reports \
  --name partner-read-2025 \
  --permissions r \
  --expiry 2025-12-31T23:59Z

# Revoke it (all SAS signed with si=partner-read-2025 stop working immediately)
az storage container policy delete --account-name mysa -c reports -n partner-read-2025
```

In [ ]:
# Demonstrate revocable vs non-revocable SAS
STORED_POLICIES = {
    'partner-read-2025': {'permissions': 'r', 'expiry': '2025-12-31'},
    'etl-write-daily':   {'permissions': 'rw','expiry': '2025-06-30'},
}

def validate_sas(sas):
    # Simplified: real validation also checks times, IP, signature, etc.
    if sas.get('si'):
        policy = STORED_POLICIES.get(sas['si'])
        if policy is None:
            return '❌ 403 — stored access policy not found (revoked!)'
        return f'✅ allowed with policy {sas["si"]} ({policy["permissions"]}, until {policy["expiry"]})'
    if sas.get('se') < '2030-01-01':        # ad-hoc SAS, still within expiry
        return '✅ allowed (ad-hoc SAS — cannot be revoked without rotating keys)'
    return '❌ expired'

ad_hoc  = {'sp': 'r', 'se': '2025-07-01'}
policy_backed = {'si': 'partner-read-2025', 'se': '2025-12-31'}

print('Before revocation:')
print(' ad-hoc       ', validate_sas(ad_hoc))
print(' policy-backed', validate_sas(policy_backed))

# Attacker leaks the partner SAS — we revoke the policy
del STORED_POLICIES['partner-read-2025']
print('\nAfter deleting the stored access policy:')
print(' ad-hoc       ', validate_sas(ad_hoc),          '  ← still valid, scary!')
print(' policy-backed', validate_sas(policy_backed),   '  ← dead instantly ✅')

## 4. Network protection — firewall & private endpoints

By default, a new storage account accepts connections **from the Internet** as
long as the caller has a valid key/SAS/Entra token. That is rarely what you
want for production data.

```bash
# Default = Deny, allow only specific VNet subnets
az storage account update -g rg-prod -n mysa \
  --default-action Deny \
  --bypass AzureServices                       # let Azure Backup / Monitor in

# Allow a specific subnet (service endpoint)
az storage account network-rule add -g rg-prod --account-name mysa \
  --vnet-name vnet-prod --subnet subnet-app

# Or — the best option — a Private Endpoint (the account gets a private IP)
az network private-endpoint create -g rg-prod -n pe-mysa-blob \
  --vnet-name vnet-prod --subnet subnet-pe \
  --private-connection-resource-id /subscriptions/.../storageAccounts/mysa \
  --group-id blob \
  --connection-name pe-mysa-blob-conn
```

| Option | Good for | Gotcha |
|--------|---------|--------|
| **Service endpoint**   | Subnets in the same region   | Traffic still uses Microsoft backbone, not your VNet IP space |
| **Private endpoint**   | All hybrid / on-prem / peered scenarios | Needs private DNS zone for the hostname to resolve |
| **`--bypass AzureServices`** | Backup, Monitor, Defender   | Overly broad if you don't need it |

### Public network access vs firewall — know the difference
- `--public-network-enabled false` → account is **only** reachable via private endpoint. Even the Azure Portal can't browse it from the Internet.
- `--default-action Deny` → Internet clients blocked unless in the IP/VNet allowlist. Private endpoint still works.

In [ ]:
# Evaluate whether a request would be allowed by the storage firewall
ACCOUNT = {
    'public_network_enabled': True,     # master switch
    'default_action': 'Deny',           # Deny or Allow
    'ip_allowlist': ['203.0.113.0/24'],
    'vnet_allowlist': ['subnet-app'],   # subnets with service endpoint
    'bypass': ['AzureServices'],
    'private_endpoint': True,
}

def ip_in(ip, prefix):
    import ipaddress
    return ipaddress.ip_address(ip) in ipaddress.ip_network(prefix)

def allow(request):
    if request.get('via_private_endpoint') and ACCOUNT['private_endpoint']:
        return '✅ via private endpoint — firewall does not apply'
    if not ACCOUNT['public_network_enabled']:
        return '❌ public network disabled — only private endpoint works'
    if request.get('from_azure_service') and 'AzureServices' in ACCOUNT['bypass']:
        return '✅ trusted Azure service bypass'
    if request.get('subnet') in ACCOUNT['vnet_allowlist']:
        return '✅ allowed by VNet rule'
    if request.get('src_ip') and any(ip_in(request['src_ip'], p) for p in ACCOUNT['ip_allowlist']):
        return '✅ allowed by IP rule'
    if ACCOUNT['default_action'] == 'Allow':
        return '⚠️ allowed by default-allow (dangerous)'
    return '❌ blocked by firewall default-deny'

requests = [
    {'label': 'developer at coffee shop',    'src_ip': '185.1.2.3'},
    {'label': 'office building static IP',   'src_ip': '203.0.113.42'},
    {'label': 'VM in subnet-app',            'subnet': 'subnet-app'},
    {'label': 'Azure Backup service',        'from_azure_service': True},
    {'label': 'app via private endpoint',    'via_private_endpoint': True},
]
for r in requests:
    print(f'{r["label"]:30s} → {allow(r)}')

## 5. Encryption — layers you can stack

| Layer | What it does | When to use |
|-------|-------------|-------------|
| **SSE (default)**                  | AES-256 at rest, platform-managed key | Always on — nothing to configure |
| **SSE with CMK (BYOK)**            | Your key in Key Vault / Managed HSM   | Regulatory need for key control / kill switch |
| **Infrastructure encryption**      | Second independent 256-bit layer      | FIPS / DoD scenarios needing double encryption |
| **Client-side encryption**         | App encrypts *before* upload          | Zero-trust: Azure never sees plaintext |

```bash
# BYOK requirements:
#   - Key Vault must have soft delete + purge protection
#   - Storage account identity must have "Key Vault Crypto Service Encryption User"
az keyvault key create --vault-name my-kv -n storage-cmk --kty RSA --size 3072

az storage account update -g rg-prod -n mysa \
  --encryption-key-source Microsoft.Keyvault \
  --encryption-key-vault https://my-kv.vault.azure.net \
  --encryption-key-name storage-cmk

# Infrastructure encryption — MUST be enabled at creation, cannot be turned on later!
az storage account create -g rg-prod -n mysa2 -l eastus --sku Standard_GRS \
  --require-infrastructure-encryption
```

> **Exam trap**: Infrastructure encryption is **create-time only**. If the
> question says "enable double encryption on an existing account" the correct
> answer is "you can't — create a new account and copy data".

## 6. Data protection — soft delete, versioning, immutable storage

| Feature                  | What it protects against | CLI |
|--------------------------|-------------------------|-----|
| **Blob soft delete**     | Accidental blob delete (recover for N days) | `--enable-delete-retention true --delete-retention-days 30` |
| **Container soft delete**| Accidental container delete | `--enable-container-delete-retention true --container-delete-retention-days 30` |
| **Versioning**           | Accidental overwrite; every PUT keeps history | `--enable-versioning true` |
| **Point-in-time restore**| Ransomware / bulk corruption; rewind account to time T | `--enable-restore-policy true --restore-days 30` (needs versioning + soft-delete + change-feed) |
| **Immutable storage (WORM)** | Compliance: nobody can delete/modify — even the subscription owner | Container- or version-level immutability |
| **Object replication**   | Geo redundancy + off-region copy | `az storage account or-policy create ...` |

### Immutable storage — two flavours
- **Time-based retention**: writes are permanent for N days. Good for regulated data.
- **Legal hold**: no time limit. Cleared only when the hold is removed. Good for lawsuits.

```bash
# Lock compliance data for 7 years
az storage container immutability-policy create -g rg-prod \
  --account-name mysa -c compliance \
  --period 2555 --allow-protected-append-writes true

# Lock the policy itself (can no longer be shortened — only extended)
az storage container immutability-policy lock -g rg-prod \
  --account-name mysa -c compliance
```

In [ ]:
# Simulate blob operations with soft delete + versioning + immutability
from datetime import datetime

class ProtectedContainer:
    def __init__(self, soft_delete_days=7, versioning=False, immutable_days=0):
        self.soft_delete_days = soft_delete_days
        self.versioning = versioning
        self.immutable_days = immutable_days     # time-based WORM
        self.blobs = {}        # path -> [versions]
        self.trash = {}        # path -> (blob, deleted_at)
        self.now = datetime(2025, 1, 15)
    def put(self, path, data):
        ver = {'data': data, 'ts': self.now}
        if path in self.blobs and not self.versioning:
            # overwriting without versioning = old data gone forever
            self.blobs[path] = [ver]
            return 'overwrite (no versioning) — prior data LOST'
        if self.immutable_days and path in self.blobs:
            return '❌ immutable policy — cannot overwrite'
        self.blobs.setdefault(path, []).append(ver)
        return f'stored version #{len(self.blobs[path])}'
    def delete(self, path):
        if self.immutable_days:
            return '❌ immutable policy — cannot delete'
        blob = self.blobs.pop(path, None)
        if blob is None:
            return 'no such blob'
        self.trash[path] = (blob, self.now)
        return f'soft-deleted, recoverable for {self.soft_delete_days}d'
    def undelete(self, path):
        entry = self.trash.pop(path, None)
        if entry is None:
            return 'nothing to undelete'
        self.blobs[path] = entry[0]
        return '✅ restored'

print('--- Plain account (no protection) ---')
c = ProtectedContainer(soft_delete_days=0, versioning=False)
print(c.put('report.csv', b'v1'))
print(c.put('report.csv', b'v2'))
print('  versions kept:', len(c.blobs['report.csv']))

print('\n--- With versioning + soft delete ---')
c = ProtectedContainer(soft_delete_days=7, versioning=True)
print(c.put('report.csv', b'v1'))
print(c.put('report.csv', b'v2'))
print('  versions kept:', len(c.blobs['report.csv']))
print(' delete:', c.delete('report.csv'))
print(' undelete:', c.undelete('report.csv'))

print('\n--- With immutable (WORM) policy ---')
c = ProtectedContainer(soft_delete_days=7, versioning=True, immutable_days=2555)
print(c.put('audit.log', b'entry-1'))
print('overwrite:', c.put('audit.log', b'tampered'))
print('delete:   ', c.delete('audit.log'))

## 7. Bad → best: hardening a storage account

```bash
# ❌ BAD — everything default
az storage account create -g rg-prod -n mysabad --sku Standard_LRS

# 🟡 BETTER — encryption, soft delete, TLS 1.2
az storage account update -g rg-prod -n mysabetter \
  --min-tls-version TLS1_2 \
  --https-only true
az storage account blob-service-properties update \
  --account-name mysabetter \
  --enable-delete-retention true --delete-retention-days 30 \
  --enable-versioning true

# ✅ BEST — private endpoint, no shared keys, CMK, firewall default deny, WORM
az storage account update -g rg-prod -n mysabest \
  --allow-shared-key-access false \
  --allow-blob-public-access false \
  --public-network-enabled false \
  --default-action Deny \
  --min-tls-version TLS1_2 \
  --encryption-key-source Microsoft.Keyvault \
  --encryption-key-vault https://my-kv.vault.azure.net \
  --encryption-key-name sa-cmk
```

---
## Summary

| Control | Best-practice setting |
|---------|----------------------|
| Authentication           | Entra ID + RBAC, managed identities; disable shared key auth |
| SAS                      | User-delegation SAS, short expiry, stored access policy |
| Network                  | Public network disabled; private endpoint; TLS 1.2+ |
| Encryption               | CMK in Key Vault; infrastructure encryption at creation |
| Data protection          | Versioning + soft delete + PITR for apps; WORM for compliance |
| Logging                  | Diagnostic settings → Log Analytics (covered in Lab 4) |

**Next**: [Notebook 3 — Database Security](03_database_security.ipynb)